# Data importing and basic data cleaning: `trends.csv`

**Dataset.** Google Year in Search rankings. 26,955 rows, five columns, 83
locations, years 2001 to 2020. Each row is one entry in a top-5 list.

**Goal.** Take the raw file and produce a cleaned dataset that is safe to
analyse. Handle missing values, duplicates, wrong data types and inconsistent
entries, rename what needs renaming, fix formatting and verify the result.

Each step reports what it found and what it changed. A step that finds nothing
says so instead of inventing work.

## 1. Import the dataset

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
print('pandas', pd.__version__)

pandas 2.2.3


In [2]:
raw = pd.read_csv('trends.csv')
df = raw.copy()

print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

Shape: (26955, 5)
Columns: ['location', 'year', 'category', 'rank', 'query']


,location,year,category,rank,query
0,Global,2001,Consumer Brands,1,Nokia
1,Global,2001,Consumer Brands,2,Sony
2,Global,2001,Consumer Brands,3,BMW
3,Global,2001,Consumer Brands,4,Palm
4,Global,2001,Consumer Brands,5,Adobe


The file loads with no parser errors. `raw` is kept untouched as a reference so
that every later step can be compared against the original.

Five columns. `location`, `year`, `category`, `rank`, `query`. Each row is one
entry in a ranked top-5 list.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26955 entries, 0 to 26954
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   location  26955 non-null  object
 1   year      26955 non-null  int64 
 2   category  26955 non-null  object
 3   rank      26955 non-null  int64 
 4   query     26955 non-null  object
dtypes: int64(2), object(3)
memory usage: 1.0+ MB


26,955 non-null entries in all five columns. Memory around 1 MB. `year` and
`rank` arrive as `int64` and the three text columns as `object`.

## 2. Identify and handle missing values

In [4]:
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Total missing:', df.isnull().sum().sum())

Missing values per column:
location    0
year        0
category    0
rank        0
query       0
dtype: int64

Total missing: 0


In [5]:
# A zero count can hide nulls stored as text. Check the usual disguises.
placeholders = {'', 'na', 'n/a', 'nan', 'null', 'none', '-', '--', 'unknown', '?'}

for col in ['location', 'category', 'query']:
    s = df[col].astype(str)
    blank = (s.str.strip() == '').sum()
    fake  = s.str.strip().str.lower().isin(placeholders).sum()
    print(f'{col:9s} blank/whitespace-only: {blank:4d}   placeholder text: {fake:4d}')

location  blank/whitespace-only:    0   placeholder text:    0
category  blank/whitespace-only:    0   placeholder text:    0
query     blank/whitespace-only:    0   placeholder text:    0


No missing values, and none disguised as blanks or placeholder text.

**Action taken: none.** There is nothing to impute or drop. The file is a
published, curated ranking rather than raw collected data, so no value ever had
the chance to go missing. Imputing here would invent data.

## 3. Remove duplicate records

In [6]:
dup_count = df.duplicated().sum()
print('Fully duplicated rows:', dup_count)
print('Share of dataset:', round(100 * dup_count / len(df), 4), '%')

df[df.duplicated(keep=False)].sort_values(['location','year','category','rank'])

Fully duplicated rows: 10
Share of dataset: 0.0371 %


,location,year,category,rank,query
19995,Kazakhstan,2018,Жылдың фильмі/Фильм года,1,Веном
20000,Kazakhstan,2018,Жылдың фильмі/Фильм года,1,Веном
19996,Kazakhstan,2018,Жылдың фильмі/Фильм года,2,Мстители: Война бесконечности
20001,Kazakhstan,2018,Жылдың фильмі/Фильм года,2,Мстители: Война бесконечности
19997,Kazakhstan,2018,Жылдың фильмі/Фильм года,3,Бизнес по-казахски в Америке
20002,Kazakhstan,2018,Жылдың фильмі/Фильм года,3,Бизнес по-казахски в Америке
19998,Kazakhstan,2018,Жылдың фильмі/Фильм года,4,Монстры на каникулах 3
20003,Kazakhstan,2018,Жылдың фильмі/Фильм года,4,Монстры на каникулах 3
19999,Kazakhstan,2018,Жылдың фильмі/Фильм года,5,Дэдпул 2
20004,Kazakhstan,2018,Жылдың фильмі/Фильм года,5,Дэдпул 2


Ten duplicated rows, and they are not scattered. They form two complete top-5
blocks, each stored twice:

1. Kazakhstan, 2018, `Жылдың фильмі/Фильм года`, ranks 1 to 5.
2. Kenya, 2020, `Trending How To  (Tech)`, ranks 1 to 5.

Two whole blocks repeated cleanly points at an append error during file
assembly rather than sloppy data entry. The rows match on all five columns, so
dropping them is safe and loses nothing.

In [7]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Rows before: {before}')
print(f'Rows after : {len(df)}')
print(f'Removed    : {before - len(df)}')
print('Duplicates remaining:', df.duplicated().sum())

Rows before: 26955
Rows after : 26945
Removed    : 10


Duplicates remaining: 0


## 4. Correct inconsistent or incorrect entries

Three separate problems live in the text columns. Whitespace, capitalisation and
the category labels themselves.

### 4a. Whitespace

In [8]:
for col in ['location', 'category', 'query']:
    s = df[col].astype(str)
    print(f'{col:9s} leading/trailing: {(s != s.str.strip()).sum():4d}   '
          f'internal double spaces: {s.str.contains("  ", regex=False).sum():4d}')

location  leading/trailing:    0   internal double spaces:    0
category  leading/trailing:  175   internal double spaces:   25
query     leading/trailing:    0   internal double spaces:    4


In [9]:
for col in ['location', 'category', 'query']:
    df[col] = (df[col].astype(str)
                      .str.strip()
                      .str.replace(r'\s+', ' ', regex=True))

print('After cleaning:')
for col in ['location', 'category', 'query']:
    s = df[col]
    print(f'{col:9s} leading/trailing: {(s != s.str.strip()).sum():4d}   '
          f'internal double spaces: {s.str.contains("  ", regex=False).sum():4d}')
print()
print('Distinct categories before whitespace fix:', raw.category.nunique())
print('Distinct categories after  whitespace fix:', df.category.nunique())

After cleaning:
location  leading/trailing:    0   internal double spaces:    0
category  leading/trailing:    0   internal double spaces:    0
query     leading/trailing:    0   internal double spaces:    0



Distinct categories before whitespace fix: 2450
Distinct categories after  whitespace fix: 2443


175 category values carried leading or trailing spaces, 25 had internal
double spaces and 4 query values did too, `Trending How To  (Tech)` among them. Stripping and collapsing
whitespace removed 7 phantom categories that existed only because of spacing.

Whitespace is fixed first on purpose. It inflates every unique count, so any
later deduplication of meaning would otherwise start from a wrong number.

### 4b. Category labels in local languages

In [10]:
print('Distinct categories:', df.category.nunique())
cat_locs = df.groupby('category')['location'].nunique()
print('Categories used in only one location:', (cat_locs == 1).sum())
print()
print(df.category.value_counts().head(12).to_string())

Distinct categories: 2443
Categories used in only one location: 2126

category
People                     760
Searches                   620
Movies                     330
TV Shows                   305
Películas                  250
Songs                      215
Fastest Rising Searches    190
What is...?                175
Recipes                    175
How to...                  170
News                       150
Events                     125


2,443 distinct categories against 83 locations, and 87 percent of them appear in
a single country. The cause is visible in the counts. `Movies` has 330 rows and
`Películas`, the same word in Spanish, sits separately with 250. `Athletes` and
`Deportistas` are the same concept. So are `How to...` and `Cómo`.

Each country recorded its labels in its own language, which means one concept is
split across many strings, which means `groupby('category')` returns over two
thousand groups and any cross-country comparison built on it is meaningless.

The fix is a rule-based mapper. A lookup table would need 2,443 hand-written
entries, so instead each raw label is matched against keyword patterns covering
the main concepts across the languages present.

In [11]:
RULES = [
 ('How To',       r'how ?to|c\u00f3mo|^como|wie |hur |jak |nas\u0131l|l\u00e0m|c\u00e1ch|\u5982\u4f55|\u600e\u4e48|\u0643\u064a\u0641'),
 ('What Is / Questions', r'what is|qu\u00e9 es|que es|^qu\u00e9|^que\b|was ist|l\u00e0 g\u00ec|nedir|\u0447\u0442\u043e \u0442\u0430\u043a\u043e\u0435|\u4ec0\u4e48|question|pregunta|sorular|\u0645\u0627\u0647\u0648|\u0645\u0627 \u0647\u0648'),
 ('Why',          r'^why|por qu\u00e9|porque|pourquoi|warum|neden|\u043f\u043e\u0447\u0435\u043c\u0443|\u0644\u0645\u0627\u0630\u0627'),
 ('Movies',       r'movie|film|pel[\u00edi]cula|\u043a\u0438\u043d\u043e|\u6620\u753b|\uc601\ud654|\u0623\u0641\u0644\u0627\u0645'),
 ('TV Shows',     r'\btv\b|programas de tv|series|dizi|s\u00e9ries|\u0442\u0435\u043b\u0435\u043f\u0435\u0440\u0435\u0434\u0430\u0447|\u0645\u0633\u0644\u0633\u0644\u0627\u062a'),
 ('Music',        r'song|music|m\u00fasica|musica|canci[o\u00f3]n|canciones|\u015fark\u0131|\u043c\u0443\u0437\u044b\u043a|chanson|lied|lyric|letra|cantante|singer|piosenk|\u0623\u063a\u0627\u0646\u064a|\u0623\u063a\u0646\u064a\u0629|m\u00fczik'),
 ('People',       r'people|person|\u043b\u044e\u0434\u0438|ki\u015filer|personnalit|pessoas|\u4eba\u7269|celebrit|famous|ludzie|\u0634\u062e\u0635\u064a\u0627\u062a|personalidad|persoane'),
 ('Athletes',     r'athlet|deportista|sporcu|sportif|sportow'),
 ('Sports',       r'sport|deporte|spor\b|\u0441\u043f\u043e\u0440\u0442|\u0631\u064a\u0627\u0636\u0629'),
 ('Searches',     r'search|b\u00fasqueda|busqueda|recherche|arama|suchbegriffe|\u043f\u043e\u0438\u0441\u043a|\u0437\u0430\u043f\u0438\u0442|trending|tendencia|emergentes|most popular|overall|fastest rising|general|\u0623\u0628\u062d\u0627\u062b|wyszukiw'),
 ('Recipes',      r'recipe|receta|recette|rezept|tarif|\u0440\u0435\u0446\u0435\u043f\u0442|resep|ricett|przepis|\u0648\u0635\u0641\u0627\u062a'),
 ('News',         r'news|noticia|nachrichten|haber|\u65b0\u95fb|\u0623\u062e\u0628\u0627\u0631'),
 ('Events',       r'event|acontecimiento|evento|olay|\u0441\u043e\u0431\u044b\u0442|wydarzen'),
 ('Losses',       r'^loss|losses|obituar|fallecid|vefat|memoriam|zmarli'),
 ('Games',        r'game|juego|spiele|oyun|\u0438\u0433\u0440|gry\b'),
 ('Actors',       r'actor|actriz|actress|oyuncu|aktor'),
 ('Politicians',  r'politic|pol\u00edtico|politiker|siyaset|polityc'),
 ('Travel',       r'travel|destino|destination|seyahat|\u043f\u0443\u0442\u0435\u0448\u0435\u0441\u0442\u0432|tatil|podr\u00f3\u017c'),
 ('Food & Drink', r'food|drink|comida|bebida|yemek|\u0435\u0434\u0430|jedzenie'),
 ('Health',       r'diet|health|salud|sa\u011fl\u0131k|\u0437\u0434\u043e\u0440\u043e\u0432|fitness|zdrow'),
 ('Technology',   r'gadget|tech|tecnolog|teknoloji|tel\u00e9fono|phone|app\b|software|\u062a\u0642\u0646\u064a\u0629'),
 ('Memes',        r'meme'),
 ('Entertainment',r'entertain|entretenimiento|rozrywka'),
 ('Near Me',      r'near me|cerca de m\u00ed|yak\u0131n'),
]
COMPILED = [(lab, re.compile(pat)) for lab, pat in RULES]

def standardise_category(value):
    s = str(value).strip().lower()
    for lab, rx in COMPILED:
        if rx.search(s):
            return lab
    return 'Unmapped'

df['category_standard'] = df['category'].map(standardise_category)

coverage = (df.category_standard != 'Unmapped').mean() * 100
print(f'Standard categories created: {df.category_standard.nunique() - 1}')
print(f'Rows mapped to a standard label: {coverage:.1f}%')
print()
print(df.category_standard.value_counts().to_string())

Standard categories created: 24
Rows mapped to a standard label: 55.6%

category_standard
Unmapped               11960
Searches                2890
People                  2300
Movies                  1240
Music                   1120
TV Shows                1075
What Is / Questions     1050
How To                   910
Sports                   670
Events                   670
News                     500
Recipes                  430
Technology               385
Athletes                 315
Politicians              215
Games                    180
Losses                   180
Travel                   160
Health                   155
Actors                   125
Food & Drink             120
Why                      110
Memes                     80
Entertainment             60
Near Me                   45


2,443 raw labels collapse to 24 standard categories covering 55.6 percent of
rows.

The remaining 44 percent stays as `Unmapped`. That is deliberate and worth
stating plainly rather than hiding. The tail is written in Arabic, Ukrainian,
Thai, Hebrew, Korean and others, and inventing translations to inflate the
coverage number would put wrong labels into the dataset. The original
`category` column is kept untouched next to the new one, so nothing is lost and
the mapping can be extended later.

Use `category_standard` for cross-country comparison. Use `category` when the
original wording matters.

### 4c. Capitalisation

In [12]:
for col in ['category', 'query']:
    s = df[col]
    print(f'{col:9s} distinct: {s.nunique():6d}   case-insensitive distinct: {s.str.lower().nunique():6d}   '
          f'differ only by case: {s.nunique() - s.str.lower().nunique():4d}')

category  distinct:   2443   case-insensitive distinct:   2366   differ only by case:   77
query     distinct:  18430   case-insensitive distinct:  17984   differ only by case:  446


76 category values and 446 query values differ from another value only by
capitalisation.

**Action taken: none on `query`.** Capitalisation carries meaning in search
terms. `Apple` the company and `apple` the fruit are different queries, and a
brand such as `iPhone` would be damaged by forcing case. The `category_standard`
column already solves the grouping problem for categories, and it lowercases
internally before matching, so case no longer blocks any analysis. Folding case
in the raw columns would destroy information to fix a problem that is already
handled.

## 5. Assign appropriate data types

In [13]:
print('Before:')
print(df.dtypes)
print()
print('Memory (MB):', round(df.memory_usage(deep=True).sum() / 1024**2, 2))

Before:
location             object
year                  int64
category             object
rank                  int64
query                object
category_standard    object
dtype: object

Memory (MB): 8.21


In [14]:
from pandas.api.types import CategoricalDtype

rank_type = CategoricalDtype(categories=[1, 2, 3, 4, 5], ordered=True)

df['location']          = df['location'].astype('category')
df['category']          = df['category'].astype('category')
df['category_standard'] = df['category_standard'].astype('category')
df['query']             = df['query'].astype('string')
df['year']              = df['year'].astype('int16')
df['rank']              = df['rank'].astype(rank_type)

print('After:')
print(df.dtypes)
print()
print('Memory (MB):', round(df.memory_usage(deep=True).sum() / 1024**2, 2))

After:
location                   category
year                          int16
category                   category
rank                       category
query                string[python]
category_standard          category
dtype: object

Memory (MB): 2.69


Each change has a reason.

`location`, `category` and `category_standard` become `category` dtype. All three
hold few distinct values against 26,945 rows, so the strings are stored once and
referenced by code.

`year` becomes `int16`. The range 2001 to 2020 fits comfortably and `int64` wastes
six bytes per row.

`rank` becomes an **ordered categorical** limited to 1 through 5. This is the
important one. As an integer, `df['rank'].mean()` returns 3.0 and looks like a
result, when rank is an ordinal position and its mean is fixed by construction.
An ordered categorical keeps the sort order working while making the meaningless
arithmetic impossible.

`query` becomes `string` rather than `object`, which is the correct modern dtype
for text.

Memory drops sharply because the category dtype removes the repeated strings.

## 6. Rename columns

In [15]:
print('Does `rank` collide with a DataFrame method?', hasattr(pd.DataFrame, 'rank'))

df = df.rename(columns={'rank': 'rank_position'})
print('Columns now:', list(df.columns))

Does `rank` collide with a DataFrame method? True
Columns now: ['location', 'year', 'category', 'rank_position', 'query', 'category_standard']


One rename, for a real reason. `rank` is also a `DataFrame` method, so
`df.rank` returns the method rather than the column and only `df['rank']` works.
That is a genuine trap during analysis, and `rank_position` also describes the
column better.

The other four names are already lowercase, descriptive and free of spaces, so
renaming them would be churn.

In [16]:
# Separate the Global aggregate from individual countries.
df['is_global'] = df['location'] == 'Global'

print(df.groupby('is_global', observed=True).size().rename('rows').to_string())

is_global
False    25810
True      1135


`location` mixes two levels of granularity. `Global` is an aggregate, not a
country, and it sits in the same column as the 82 real countries. Any
`groupby('location')` therefore double-counts, once inside `Global` and once
inside the country that also reported the query.

Nothing in the raw data marks it as different, so `is_global` makes the
distinction explicit. Filter with `df[~df.is_global]` for country-level work.

## 7. Check and correct formatting issues

In [17]:
checks = {
 'leading/trailing whitespace': sum((df[c].astype(str) != df[c].astype(str).str.strip()).sum()
                                    for c in ['location','category','query']),
 'internal double spaces'     : sum(df[c].astype(str).str.contains('  ', regex=False).sum()
                                    for c in ['location','category','query']),
 'empty strings'              : sum((df[c].astype(str).str.strip() == '').sum()
                                    for c in ['location','category','query']),
 'year outside 2001-2020'     : int((~df.year.between(2001, 2020)).sum()),
 'rank outside 1-5'           : int(df.rank_position.isna().sum()),
}
for k, v in checks.items():
    print(f'{k:30s} {v}')

leading/trailing whitespace    0
internal double spaces         0
empty strings                  0
year outside 2001-2020         0
rank outside 1-5               0


In [18]:
# Put the columns in a readable order and sort the rows predictably.
df = df[['location', 'is_global', 'year', 'category', 'category_standard',
         'rank_position', 'query']]

df = df.sort_values(['location', 'year', 'category', 'rank_position']).reset_index(drop=True)
df.head(10)

,location,is_global,year,category,category_standard,rank_position,query
0,Argentina,False,2008,Conflicto del Campo,Unmapped,1,Retenciones
1,Argentina,False,2008,Conflicto del Campo,Unmapped,2,Campo vs. Gobierno
2,Argentina,False,2008,Conflicto del Campo,Unmapped,3,Kirchner
3,Argentina,False,2008,Conflicto del Campo,Unmapped,4,Alfredo De Angelis
4,Argentina,False,2008,Conflicto del Campo,Unmapped,5,Cobos
5,Argentina,False,2008,Economia,Unmapped,1,Dolar
6,Argentina,False,2008,Economia,Unmapped,2,PBI
7,Argentina,False,2008,Economia,Unmapped,3,Rentas
8,Argentina,False,2008,Economia,Unmapped,4,Inflacion
9,Argentina,False,2008,Economia,Unmapped,5,Crisis


Every formatting check returns zero. `rank_position` has no nulls, which also
confirms that no value fell outside 1 to 5 when it was cast to the ordered
categorical, since an out-of-range value would have become `NaN`.

Columns are reordered so the identifying fields come first and the query last,
and rows are sorted by location, year, category and rank so the file reads
predictably.

## 8. Verify the final cleaned dataset

In [19]:
print('=== FINAL CHECKS ===')
print(f'Rows      : {len(raw)} raw  ->  {len(df)} cleaned   ({len(raw) - len(df)} removed)')
print(f'Columns   : {raw.shape[1]} raw  ->  {df.shape[1]} cleaned   (+2 derived)')
print(f'Missing   : {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print()
print(df.dtypes.to_string())

=== FINAL CHECKS ===
Rows      : 26955 raw  ->  26945 cleaned   (10 removed)
Columns   : 5 raw  ->  7 cleaned   (+2 derived)
Missing   : 0
Duplicates: 0

location                   category
is_global                      bool
year                          int16
category                   category
category_standard          category
rank_position              category
query                string[python]


In [20]:
# The dataset's structural rule: one top-5 list per location, year and category.
sizes = df.groupby(['location', 'year', 'category'], observed=True).size()

print('Total groups            :', len(sizes))
print('Groups with exactly 5   :', (sizes == 5).sum())
print('Groups NOT equal to 5   :', (sizes != 5).sum())
print()
bad_rank = df.groupby(['location','year','category'], observed=True)['rank_position'] \
             .apply(lambda s: s.duplicated().sum())
print('Groups with a repeated rank:', (bad_rank > 0).sum())

Total groups            : 5389
Groups with exactly 5   : 5389
Groups NOT equal to 5   : 0



Groups with a repeated rank: 0


Every one of the 5,389 groups now holds exactly five rows and no group repeats a
rank. Before cleaning, two groups held ten. The structural rule holds across the
whole file, which confirms the duplicate removal took out the right rows and
nothing else.

In [21]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
location,26945,83,United States,2070,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_global,26945,2,False,25810,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,26945.0,NaN,NaN,NaN,2015.241974,3.564557,2001.0,2013.0,2016.0,2018.0,2020.0
category,26945,2443,People,760,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category_standard,26945,25,Unmapped,11960,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rank_position,26945.0,5.0,1.0,5389.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
query,26945,18430,Paul Walker,84,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
df.to_csv('trends_cleaned.csv', index=False)

check = pd.read_csv('trends_cleaned.csv')
print('Written  : trends_cleaned.csv')
print('Re-read  :', check.shape)
print('Matches  :', check.shape[0] == df.shape[0] and check.shape[1] == df.shape[1])
check.head()

Written  : trends_cleaned.csv
Re-read  : (26945, 7)
Matches  : True


,location,is_global,year,category,category_standard,rank_position,query
0,Argentina,False,2008,Conflicto del Campo,Unmapped,1,Retenciones
1,Argentina,False,2008,Conflicto del Campo,Unmapped,2,Campo vs. Gobierno
2,Argentina,False,2008,Conflicto del Campo,Unmapped,3,Kirchner
3,Argentina,False,2008,Conflicto del Campo,Unmapped,4,Alfredo De Angelis
4,Argentina,False,2008,Conflicto del Campo,Unmapped,5,Cobos


## Summary of cleaning steps

| # | Step | Found | Action |
|---|------|-------|--------|
| 1 | Import | 26,955 rows, 5 columns, no parser errors | Loaded, kept `raw` copy for comparison |
| 2 | Missing values | 0 missing, 0 disguised as blanks or placeholders | None needed |
| 3 | Duplicates | 10 rows, forming 2 complete top-5 blocks | Dropped, 26,945 remain |
| 4a | Whitespace | 175 values with stray spaces, 29 with double spaces | Stripped and collapsed, removed 7 phantom categories |
| 4b | Category languages | 2,443 labels, 87% single-country | Added `category_standard`, 24 labels, 55.6% of rows |
| 4c | Capitalisation | 76 category and 446 query case-variants | Left as is, case carries meaning in search terms |
| 5 | Data types | All text as `object`, `rank` as `int64` | Cast to `category`, `string`, `int16`, ordered categorical |
| 6 | Column names | `rank` shadows `DataFrame.rank` | Renamed to `rank_position`, added `is_global` |
| 7 | Formatting | All checks zero after 4a | Reordered columns, sorted rows |
| 8 | Verification | 5,389 groups, all exactly 5 rows | Exported `trends_cleaned.csv` |

**Result.** 26,945 rows and 7 columns. No missing values, no duplicates, correct
data types and a standardised category column usable for cross-country grouping.

**Known limit.** 44 percent of rows keep `Unmapped` in `category_standard`. The
tail of category labels is written in scripts the keyword rules do not cover, and
guessing translations would put wrong labels into the data. The original
`category` column is preserved, so the mapping can be extended without redoing
any of this work.